In [14]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing

housing=fetch_california_housing()

In [15]:
##Preparing the dataset
data=pd.DataFrame(housing.data,columns=housing.feature_names)
data["Price"]=housing.target

In [16]:
from urllib.parse import urlparse
##Independent and Dependent features
X=data.drop(columns=['Price'])
y=data['Price']

In [17]:
##Hyperparameter tuning using Grid Searchcv

def hyperparameter_tuning(X_train, y_train, param_grid):
    rf=RandomForestRegressor()
    grid_search=GridSearchCV(estimator=rf,param_grid=param_grid,cv=3,n_jobs=-1,verbose=2,scoring='neg_mean_squared_error')
    grid_search.fit(X_train,y_train)
    return grid_search

In [20]:
##Split data into training and test sets
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

from mlflow.models import infer_signature
signature=infer_signature(X_train,X_test)

##Define the hyperparameter grid
param_grid={
    'n_estimators':[100,200],
    'max_depth':[5,10,None],
    'min_samples_split':[2,5],
    'min_samples_leaf':[1,2]
}

##Start the MLFlow Experiments
with mlflow.start_run():
    ##perform hyperparameter tuning
    grid_search=hyperparameter_tuning(X_train, y_train, param_grid)

    ##Get the best mode
    best_model=grid_search.best_estimator_

    ##Evaluate the best model
    y_pred=best_model.predict(X_test)
    mse=mean_squared_error(y_test,y_pred)

    ##Log best parameters and metrics
    mlflow.log_param("best_n_estimators",grid_search.best_params_["n_estimators"])
    mlflow.log_param("max_depth",grid_search.best_params_["max_depth"])
    mlflow.log_param("min_samples_split",grid_search.best_params_["min_samples_split"])
    mlflow.log_param("min_samples_leaf",grid_search.best_params_["min_samples_leaf"])
    mlflow.log_metric("mse",mse)

    ##Tracking url

    mlflow.set_tracking_uri(uri="http://127.0.0.1:5000")
    tracking_uri_type_store=urlparse(mlflow.get_tracking_uri()).scheme

    if tracking_uri_type_store!="file":
        mlflow.sklearn.log_model(best_model,"model",registered_model_name="best_Randomforest_model" )
    else:
        mlflow.sklearn.log_model(best_model,"model",signature=signature)

    print(f"Best Hyperparameters: {grid_search.best_params_}")
    print(f"Mean Squared Error: {mse}")




Fitting 3 folds for each of 24 candidates, totalling 72 fits
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.1s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.1s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   2.1s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   2.1s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   2.2s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   4.0s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   2.1s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   4.2s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   4.2s
[CV] END max_depth=5, min_samples_leaf=

2026/06/27 23:27:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'best_Randomforest_model'.
2026/06/27 23:27:42 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: best_Randomforest_model, version 1
Created version '1' of model 'best_Randomforest_model'.


Best Hyperparameters: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
Mean Squared Error: 0.24792733747011797
🏃 View run awesome-cod-760 at: http://127.0.0.1:5000/#/experiments/0/runs/da60a011b1fa4e6ea13a571ab5acdc39
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
